# 🔤Tokenization

## Mục tiêu bài học
- Hiểu cách Large Language Models (LLM) chia nhỏ văn bản thành tokens
- So sánh sự khác biệt giữa tokenization tiếng Việt và tiếng Anh
- Tính toán chi phí sử dụng API dựa trên số lượng tokens
- Tối ưu hóa prompt để tiết kiệm chi phí

## 📚 Phần 1: Giới thiệu về Tokenization

### Tokenization là gì?
Tokenization là quá trình chia nhỏ văn bản thành các đơn vị nhỏ hơn gọi là **tokens**. Đây là bước đầu tiên mà LLM thực hiện khi xử lý văn bản.

### Tại sao cần tokenization?
- LLM không xử lý trực tiếp văn bản, mà xử lý các con số (tokens)
- Mỗi token được chuyển đổi thành một vector số để model có thể hiểu
- Chi phí API được tính dựa trên số lượng tokens (input + output)

### Một số quy tắc cơ bản:
- 1 token ≈ 4 ký tự tiếng Anh
- 1 token ≈ ¾ từ tiếng Anh
- 1 từ tiếng Việt có thể tốn nhiều token hơn tiếng Anh (2-3 tokens)
- Khoảng trắng, dấu câu cũng tốn tokens

## 🛠️ Phần 2: Cài đặt và Chuẩn bị

Chúng ta sẽ sử dụng thư viện `tiktoken` - công cụ tokenization của OpenAI

In [1]:
# Cài đặt thư viện cần thiết
%pip install --upgrade tiktoken pandas
%pip install --upgrade torch --index-url https://download.pytorch.org/whl/cpu
%pip install --upgrade transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.4 MB 4.1 MB/s eta 0:00:03
   --- ------------------------------------ 0.8/10.4 MB 4.9 MB/s eta 0:00:02
   --------- ------------------------------ 2.5/10.4 MB 12.3 MB/s eta 0:00:01
   ----------- ---------------------------- 2.9/10.4 MB 12.2 MB/s eta 0:00:01
   ------------- -------------------------- 3.6/10.4 MB 11.4 MB/s eta 0:00:01
   ------------------- -------------------- 5.1/10.4 MB 14.3 MB/s eta 0:00:01
   ----------------------- ---------------- 6.1/10.4 MB 15.0 MB/s eta 0:00:01
   ------------------------------ --------- 7.9/10.4 MB 17.4 MB/s eta 0:00:01
   ------------------------------------- -- 9.8/10.4 MB 19.6 MB/s eta 0:00:01
   ---------------------------------------  10.4/10.4 MB 25.1 MB/s eta 0:00:01
   --------


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# Import thư viện
import tiktoken
import pandas as pd
from typing import List

# Import transformers với xử lý lỗi
try:
    from transformers import AutoTokenizer
    TRANSFORMERS_AVAILABLE = True
except Exception as e:
    print(f"⚠️ Không thể import transformers: {e}")
    print("📝 Hàm so sánh BPE vs WordPiece sẽ chỉ hiển thị BPE")
    TRANSFORMERS_AVAILABLE = False

## 🔬 Phần 3: Thử nghiệm Tokenization

### 3.1 Khởi tạo Tokenizer
Chúng ta sẽ sử dụng tokenizer của GPT-4 (encoding: `cl100k_base`)
và tokenizer của mô hình `google-bert/bert-base-uncased`

In [50]:
# Khởi tạo tokenizer cho GPT-4/GPT-3.5-turbo
bpe_encoding = tiktoken.get_encoding("cl100k_base")
wordpiece_tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
# Hoặc có thể khởi tạo theo tên model cụ thể
# bpe_encoding = tiktoken.encoding_for_model("gpt-4")

print("✅ Tokenizer đã được khởi tạo thành công!")

✅ Tokenizer đã được khởi tạo thành công!


### Hàm display token

In [ ]:


def display_tokens(text: str, encoding):
    """Hiển thị chi tiết tokens của văn bản và trả về mảng tokens"""
    tokens = encoding.encode(text)
    decoded = [encoding.decode([token]) for token in tokens]
    print(f"📝 Văn bản: {text}")
    print(f"   - Tokens: {decoded}")
    print(f"   - Mảng tokens: {tokens}")
    

    # Hiển thị chi tiết từng token
    print(f"\n🔍 Chi tiết tokens:")
    for i, token in enumerate(tokens):
        decoded = encoding.decode([token])
        print(f"   Token {i+1}: {token} → '{decoded}'")
    
    return tokens

### Thử nghiệm với tiếng Anh

In [51]:
# Ví dụ tiếng Anh
english_text = "hello, hello are you today?"
print("TIẾNG ANH:")
result_en = display_tokens(english_text, bpe_encoding)

TIẾNG ANH:
📝 Văn bản: hello, hello are you today?
   - Tokens: ['hello', ',', ' hello', ' are', ' you', ' today', '?']
   - Mảng tokens: [15339, 11, 24748, 527, 499, 3432, 30]

🔍 Chi tiết tokens:
   Token 1: 15339 → 'hello'
   Token 2: 11 → ','
   Token 3: 24748 → ' hello'
   Token 4: 527 → ' are'
   Token 5: 499 → ' you'
   Token 6: 3432 → ' today'
   Token 7: 30 → '?'


### Thử nghiệm với tiếng Việt

In [52]:
# Ví dụ tiếng Việt
vietnamese_text = "Xin chào, bạn khỏe không?"
print("\nTIẾNG VIỆT:")
result_vi = display_tokens(vietnamese_text, bpe_encoding)


TIẾNG VIỆT:
📝 Văn bản: Xin chào, bạn khỏe không?
   - Tokens: ['X', 'in', ' ch', 'à', 'o', ',', ' bạn', ' kh', 'ỏ', 'e', ' không', '?']
   - Mảng tokens: [55, 258, 523, 6496, 78, 11, 90537, 24040, 86242, 68, 54137, 30]

🔍 Chi tiết tokens:
   Token 1: 55 → 'X'
   Token 2: 258 → 'in'
   Token 3: 523 → ' ch'
   Token 4: 6496 → 'à'
   Token 5: 78 → 'o'
   Token 6: 11 → ','
   Token 7: 90537 → ' bạn'
   Token 8: 24040 → ' kh'
   Token 9: 86242 → 'ỏ'
   Token 10: 68 → 'e'
   Token 11: 54137 → ' không'
   Token 12: 30 → '?'


### BPE

In [ ]:
tokens = bpe_encoding.encode("unhappiness")
decoded = [bpe_encoding.decode([token]) for token in tokens]
print(f"   - Tokens: {decoded}")
print(f"   - Mảng tokens: {tokens}")

   - Tokens: ['un', 'h', 'appiness']
   - Mảng tokens: [359, 71, 67391]


### WordPiece

In [43]:
wp_tokens = wordpiece_tokenizer.encode("unhappiness")
wp_decoded = wordpiece_tokenizer.convert_ids_to_tokens(wp_tokens)
print(f"   Tokens: {wp_tokens}")
print(f"   Decoded: {wp_decoded}")

   Tokens: [101, 4895, 3270, 9397, 9961, 102]
   Decoded: ['[CLS]', 'un', '##ha', '##pp', '##iness', '[SEP]']


### So sánh thuật toán: BPE vs WordPiece

**BPE (Byte Pair Encoding):**
- Được sử dụng bởi GPT models (OpenAI)
- Chia nhỏ dựa trên các cặp byte xuất hiện nhiều nhất
- Tốt cho nhiều ngôn ngữ khác nhau

**WordPiece:**
- Được sử dụng bởi BERT models (Google)
- Chia nhỏ dựa trên xác suất tối đa hóa
- Thường sử dụng prefix `##` cho sub-word

In [65]:
def compare_tokenization_algorithms(text: str, bpe_encoding, wordpiece_tokenizer):
    # BPE tokenization
    bpe_tokens = bpe_encoding.encode(text)
    bpe_decoded = [bpe_encoding.decode([token]) for token in bpe_tokens]
    
    # WordPiece tokenization
    wp_tokens = []
    wp_decoded = []

    wp_tokens = wordpiece_tokenizer.encode(text, add_special_tokens=False)
    wp_decoded = wordpiece_tokenizer.convert_ids_to_tokens(wp_tokens)
    
    # Hiển thị kết quả
    print(f"📝 Văn bản gốc: '{text}'")
    print(f"\n{'='*60}")
    
    print(f"\n🔵 BPE- {len(bpe_tokens)} tokens:")
    print(f"   Tokens: {bpe_tokens}")
    print(f"   Decoded: {bpe_decoded}")
    

    print(f"\n🟢 WordPiece - {len(wp_tokens)} tokens:")
    print(f"   Tokens: {wp_tokens}")
    print(f"   Decoded: {wp_decoded}")
        
    print(f"\n{'='*60}")
    print(f"📊 So sánh:")
    print(f"   - BPE: {len(bpe_tokens)} tokens")
    print(f"   - WordPiece: {len(wp_tokens)} tokens")
    diff = len(bpe_tokens) - len(wp_tokens)
    print(f"   - Chênh lệch: {abs(diff)} tokens ({'BPE nhiều hơn' if diff > 0 else 'WordPiece nhiều hơn' if diff < 0 else 'Bằng nhau'})")
    
    return {
        'text': text,
        'bpe': {
            'tokens': bpe_tokens,
            'decoded': bpe_decoded,
            'count': len(bpe_tokens)
        },
        'wordpiece': {
            'tokens': wp_tokens,
            'decoded': wp_decoded,
            'count': len(wp_tokens)
        }
    }

### Ví dụ: Tiếng Anh đơn giản

In [59]:
# Ví dụ 1: Tiếng Anh đơn giản
print("VÍ DỤ 1: TIẾNG ANH ĐƠN GIẢN")
result1 = compare_tokenization_algorithms("Hello world !", bpe_encoding, wordpiece_tokenizer)

VÍ DỤ 1: TIẾNG ANH ĐƠN GIẢN
📝 Văn bản gốc: 'Hello world !'


🔵 BPE (GPT-4) - 3 tokens:
   Tokens: [9906, 1917, 758]
   Decoded: ['Hello', ' world', ' !']

🟢 WordPiece (BERT) - 3 tokens:
   Tokens: [7592, 2088, 999]
   Decoded: ['hello', 'world', '!']

📊 So sánh:
   - BPE: 3 tokens
   - WordPiece: 3 tokens
   - Chênh lệch: 0 tokens (Bằng nhau)


### Ví dụ: Từ phức tạp

In [60]:
print("\n\nVÍ DỤ 2: TỪ PHỨC TẠP")
result2 = compare_tokenization_algorithms("unhappiness", bpe_encoding=bpe_encoding, wordpiece_tokenizer=wordpiece_tokenizer)



VÍ DỤ 2: TỪ PHỨC TẠP
📝 Văn bản gốc: 'unhappiness'


🔵 BPE (GPT-4) - 3 tokens:
   Tokens: [359, 71, 67391]
   Decoded: ['un', 'h', 'appiness']

🟢 WordPiece (BERT) - 4 tokens:
   Tokens: [4895, 3270, 9397, 9961]
   Decoded: ['un', '##ha', '##pp', '##iness']

📊 So sánh:
   - BPE: 3 tokens
   - WordPiece: 4 tokens
   - Chênh lệch: 1 tokens (WordPiece nhiều hơn)


### Ví dụ: Tiếng Việt

In [63]:

print("\n\nVÍ DỤ: TIẾNG VIỆT")
result3 = compare_tokenization_algorithms("Công ty này đang làm ăn phát đạt", bpe_encoding=bpe_encoding, wordpiece_tokenizer=wordpiece_tokenizer)



VÍ DỤ: TIẾNG VIỆT
📝 Văn bản gốc: 'Công ty này đang làm ăn phát đạt'


🔵 BPE (GPT-4) - 16 tokens:
   Tokens: [34, 24976, 13892, 97635, 15199, 526, 39015, 76, 220, 6845, 77, 1343, 17099, 15199, 20842, 83]
   Decoded: ['C', 'ông', ' ty', ' này', ' đ', 'ang', ' là', 'm', ' ', 'ă', 'n', ' ph', 'át', ' đ', 'ạ', 't']

🟢 WordPiece (BERT) - 11 tokens:
   Tokens: [26478, 5939, 29349, 1102, 5654, 16983, 2019, 6887, 4017, 1102, 4017]
   Decoded: ['cong', 'ty', 'nay', 'đ', '##ang', 'lam', 'an', 'ph', '##at', 'đ', '##at']

📊 So sánh:
   - BPE: 16 tokens
   - WordPiece: 11 tokens
   - Chênh lệch: 5 tokens (BPE nhiều hơn)


### Ví dụ: Câu dài

In [66]:
print("\n\nVÍ DỤ: CÂU DÀI")
result4 = compare_tokenization_algorithms("The quick brown fox jumps over the lazy dog", bpe_encoding=bpe_encoding, wordpiece_tokenizer=wordpiece_tokenizer)



VÍ DỤ: CÂU DÀI
📝 Văn bản gốc: 'The quick brown fox jumps over the lazy dog'


🔵 BPE- 9 tokens:
   Tokens: [791, 4062, 14198, 39935, 35308, 927, 279, 16053, 5679]
   Decoded: ['The', ' quick', ' brown', ' fox', ' jumps', ' over', ' the', ' lazy', ' dog']

🟢 WordPiece - 9 tokens:
   Tokens: [1996, 4248, 2829, 4419, 14523, 2058, 1996, 13971, 3899]
   Decoded: ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']

📊 So sánh:
   - BPE: 9 tokens
   - WordPiece: 9 tokens
   - Chênh lệch: 0 tokens (Bằng nhau)


### 💡 Nhận xét về BPE vs WordPiece

**Ưu điểm BPE:**
- Linh hoạt với nhiều ngôn ngữ
- Xử lý tốt các từ hiếm
- Được sử dụng trong các model GPT hiện đại

**Ưu điểm WordPiece:**
- Tối ưu cho tiếng Anh
- Dễ phân tích với prefix `##`
- Hiệu quả cho các tác vụ NLU (Natural Language Understanding)

**Khi nào dùng gì:**
- Dùng BPE (GPT): Khi cần generate text, đa ngôn ngữ
- Dùng WordPiece (BERT): Khi cần phân tích, classification